# CIDS · 08 — S5 + S6: frequency council with dual supervision (⑥, no metadata)

**S5 — finder and checker maps (fully automatic, no ground truth):** SegResNet-100 → candidates → SAM-S2b (+ frequency adapter if given) on each candidate → **each candidate's SAM map weighted by the learned jury's P(tumour)** (S3d; training cases use a cross-fitted jury that never saw them) → two probability maps per case (1 mm) for all 1506 cases. ~2–3 h, resumable.

**S6 — the council:** 4 **independent** band experts — E1 shape (LLL) · E2 edges along one axis (LLH, LHL, HLL) · E3 corners along two axes (LHH, HLH, HHL) · E4 texture (HHH). Each expert sees **its own Haar bands of the image + the finder map + the checker map** and predicts **its own mask**. A gate combines the 4 masks into the **final mask**; their disagreement is the **uncertainty**. Dual supervision: ground truth on the final mask and on every expert + distillation towards finder and checker where they agree. ~4 h.

**Settings:** GPU T4 x2 · **Internet ON** · Persistence: Files.  
**Inputs:** cache · `cids-checkpoints` (`segresnet_img_e100_best.pt`, `sam_s2b_best.pt`, `sam_s2b_adapter_best.pt`).  
**Ablation (second account):** `USE_TEACHERS = False` (ground truth only). It can reuse the maps: attach this notebook's output and set `MAPS_DIR` to its `maps` folder, then skip S5.

In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = '/kaggle/input/datasets/kumarvemuri1/mama-mia-preprocessed-cache-2ch'
EMB_ROOT   = '/kaggle/input/datasets/kumarvemuri1/cids-sam-emb/SAM_embeddings_dataset'
CKPT_DIR   = '/kaggle/input/datasets/kumarvemuri1/cids-checkpoints'
SEG_CKPT   = f'{CKPT_DIR}/segresnet_img_e100_best.pt'
SAM_CKPT   = f'{CKPT_DIR}/sam_s2b_best.pt'
CODE_DIR   = '/kaggle/working/cids'
ADAPTER_CKPT = ''          # '' = S2b alone; after S4 (notebook 07): f'{CKPT_DIR}/sam_s2b_adapter_best.pt'
JURY         = f'{CKPT_DIR}/s3d_train_features.csv'   # learned jury (S3d) weights the checker map; '' = off
USE_TEACHERS = True
EPOCHS       = 50
MAPS_DIR     = '/kaggle/working/maps_jury'      # or a folder from an attached earlier output
RESUME_FROM  = ''
OUT = '/kaggle/working/s6_council' + ('' if USE_TEACHERS else '_gt_only')
PATHS = [CACHE_ROOT, SEG_CKPT, SAM_CKPT] + ([JURY] if JURY else []) + ([ADAPTER_CKPT] if ADAPTER_CKPT else [])


In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

def sh(cmd):
    """Run a shell step, stream its output, and STOP the notebook if it fails (plain !cmd does not).
    Progress bars redraw a line with '\r'; only each line's final state is printed (one line per epoch)."""
    import subprocess
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    buf = b''
    while True:
        chunk = p.stdout.read1(65536)
        if not chunk:
            break
        buf += chunk
        *lines, buf = buf.split(b'\n')
        for line in lines:
            print(line.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if buf.strip():
        print(buf.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}): {cmd}')


In [ ]:
sh(f'pip install -q monai medim einops appdirs')
sh(f'python -m pytest -q tests')

In [ ]:
import os, torch
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
print('GPU:', torch.cuda.get_device_properties(0).name)
for p in PATHS:
    print('ok     ' if Path(p).exists() else 'MISSING', p)
ADAPT = (f'--adapter_ckpt {ADAPTER_CKPT}' if ADAPTER_CKPT else '') + (f' --jury {JURY}' if JURY else '')
TEACH = '' if USE_TEACHERS else '--no_teachers'
resume = f'--resume_from {RESUME_FROM}' if RESUME_FROM else ''


## S5 — export finder and checker maps (~2–3 h; finished cases are skipped)
Smoke first (2 cases per split), then the full export. At the end it prints a sanity table: finder and checker Dice (rel20) on val/test — the finder should be ≈ 0.68 on test.

In [ ]:
sh(f'python scripts/s5_export_maps.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_ckpt {SAM_CKPT} {ADAPT} --out /kaggle/working/maps_smoke --limit 2')
sh(f'rm -rf /kaggle/working/maps_smoke')


In [ ]:
sh(f'python scripts/s5_export_maps.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_ckpt {SAM_CKPT} {ADAPT} --out {MAPS_DIR}')


## S6 — smoke test (8 train, 1 epoch, 4 val + 4 test, ~5 min)

In [ ]:
sh(f'python scripts/s6_freq_council.py --cache_root {CACHE_ROOT} --maps_dir {MAPS_DIR} {TEACH} --out /kaggle/working/s6_smoke --epochs 1 --val_every 1 --limit_train 8 --limit_eval 4')
sh(f'rm -rf /kaggle/working/s6_smoke')


## S6 — full run (50 epochs + test, ~4 h)

In [ ]:
sh(f'python scripts/s6_freq_council.py --cache_root {CACHE_ROOT} --maps_dir {MAPS_DIR} {TEACH} --out {OUT} --epochs {EPOCHS} {resume}')


In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f'{OUT}/train_log.csv'); v = log.dropna(subset=['val_council'])
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].plot(log.epoch, log.total, label='total'); ax[0].plot(log.epoch, log.main, label='final'); ax[0].legend(); ax[0].set_title('loss')
for c, s in (('val_council', 'o-'), ('val_finder', 's--'), ('val_checker', '^--')):
    ax[1].plot(v.epoch, v[c], s, label=c[4:])
for c in [c for c in v.columns if c.startswith('val_E')]:
    ax[1].plot(v.epoch, v[c], ':', label=c[4:])
ax[1].legend(fontsize=7); ax[1].set_title('val Dice')
for c in [c for c in v.columns if c.startswith('val_w_')]:
    ax[2].plot(v.epoch, v[c], 'o-', label=c[6:])
ax[2].legend(fontsize=7); ax[2].set_title('gate weight in tumour'); plt.tight_layout(); plt.show()
print(json.dumps(json.load(open(f'{OUT}/test_summary.json')), indent=2))
sh(f"""du -sh {OUT} {MAPS_DIR}""")
